In [ ]:
import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import joblib
import matplotlib.pyplot as plt
from utils import performance_metrics

/Users/leoinv/Documents/CODE/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/leoinv/Documents/CODE/.venv/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [ ]:
def kelly(df, f_length=10, s_length=30, fraction = 0.5):
    close = df['Close']
    rets = close.pct_change()
    f = rets.ewm(f_length).std()
    s = rets.rolling(s_length).std()
    k = f / s * fraction
    k = k.fillna(0.1)
    k = np.clip(k, 0.1, 1)
    return k

def moving_percentile_np(arr, window, percentile):
    out = np.full_like(arr, np.nan, dtype=float)
    for i in range(window - 1, len(arr)):
        window_data = arr[i - window + 1:i + 1]
        out[i] = np.percentile(window_data, percentile)
    return out

def compute_true_range_np(high, low, close):
    prev_close = np.roll(close, 1)
    prev_close[0] = close[0]
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev_close), np.abs(low - prev_close)))
    return tr

def get_session_label(ts):
    hour = ts.hour
    if 0 <= hour < 8:
        return 'Asia'
    elif 8 <= hour < 16:
        return 'Europe'
    else:
        return 'US'

def compute_session_ranges(df):
    df = df.copy()
    df['Session'] = df['Date'].map(get_session_label)
    df['SessionDate'] = df['Date'].dt.floor('D')
    df_grouped = df.groupby(['SessionDate', 'Session'])

    session_df = df_grouped.agg({
        'High': 'max',
        'Low': 'min',
        'Close': 'last'
    }).reset_index()

    high = session_df['High'].values
    low = session_df['Low'].values
    close = session_df['Close'].values
    tr = compute_true_range_np(high, low, close)

    session_df['TR'] = tr
    session_df = session_df[session_df['Session'] == 'Asia']
    session_df.set_index('SessionDate', inplace=True)
    return session_df

In [ ]:
# === Strategy ===
class MPBuyStrategy(Strategy):
    mp_window = 100
    percentile = 90
    mpfactor = 0.5

    def init(self):
        df = self.data.df.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        self.df = df
        self.session_tr = compute_session_ranges(df)
        mp_val = moving_percentile_np(self.session_tr['TR'].values, self.mp_window, self.percentile)
        self.session_tr['MP'] = mp_val
        self.session_tr = self.session_tr[['MP']]
        self.kelly = self.I(kelly, df, f_length=10, s_length=30, fraction=1)

    def next(self):
        i = len(self.data) - 1
        ts = self.df.iloc[i]['Date']
        price = self.data.Close[-1]
        session_label = get_session_label(ts)
        session_date = ts.floor('D')

        if session_label == 'Europe':
            if session_date not in self.session_tr.index:
                return
            mp_val = self.session_tr.loc[session_date, 'MP']
            if np.isnan(mp_val):
                return

            limit_price = price - mp_val * self.mpfactor
            limit_price = min(limit_price, price * (1 - 0.0025))
            size = self.kelly[-1]
            self.buy(limit=limit_price, size=size)

        elif session_label == 'Asia':
            if self.position:
                self.position.close()

In [ ]:
# === Load Data ===
ticker_data = joblib.load("/Users/leoinv/Documents/CODE/BACKTEST/data/1h.joblib")

# === Config ===
mp_window_fixed = 100
percentile_fixed = 90
mpfactor_vals = np.arange(0.5, 2, 0.2)

# === Results Containers ===
train_results = {}
test_results = {}

for ticker, df in ticker_data.items():
    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    df['Date'] = pd.to_datetime(df['Date'])

    # === Train/Test Split ===
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    best_sharpe = -np.inf
    best_mpf = None
    best_equity_train = None

    # === Optimize on Train Set ===
    for mpf in mpfactor_vals:
        MPBuyStrategy.mp_window = mp_window_fixed
        MPBuyStrategy.percentile = percentile_fixed
        MPBuyStrategy.mpfactor = mpf

        bt = Backtest(train_df, MPBuyStrategy, cash=1_000_000, commission=0.001, exclusive_orders=True)
        stats = bt.run()
        equity = stats._equity_curve['Equity']
        perf = performance_metrics(equity, risk_free_rate=0.00, periods_per_year=365*24)
        sharpe = perf['Sharpe Ratio']

        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_mpf = round(mpf, 2)
            best_equity_train = equity

    train_results[ticker] = {
        'Sharpe': best_sharpe,
        'Best mpfactor': best_mpf
    }

    # === Backtest on Test Set with Best mpfactor ===
    MPBuyStrategy.mp_window = mp_window_fixed
    MPBuyStrategy.percentile = percentile_fixed
    MPBuyStrategy.mpfactor = best_mpf

    bt = Backtest(test_df, MPBuyStrategy, cash=1_000_000, commission=0.001, exclusive_orders=True)
    stats = bt.run()
    equity_test = stats._equity_curve['Equity']
    perf_test = performance_metrics(equity_test, risk_free_rate=0.00, periods_per_year=365*24)

    test_results[ticker] = {
        'Sharpe': perf_test['Sharpe Ratio'],
        'Equity Curve': equity_test
    }

<ipython-input-4-c11d9c0ba8b7>:32: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(train_df, MPBuyStrategy, cash=1_000_000, commission=0.001, exclusive_orders=True)
<ipython-input-4-c11d9c0ba8b7>:32: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(train_df, MPBuyStrategy, cash=1_000_000, commission=0.001, exclusive_orders=True)
<ipython-input-4-c11d9c0ba8b7>:32: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(train_df, MPBuyStrategy, cash=1_000_000, commission=0.001, exclusive_orders=True)
<ipython-input-4-c11d9c0ba8b7>:32: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(train_df, MPBuyStrategy, cash=1_000_000, commission=0.001, exclusive_orders=True)
<ipython-input-4-c11d9c0ba8b7>:32: UserWarning: Data index is no

In [ ]:
# === Print Summary ===
print("=== Train Results ===")
for ticker, res in train_results.items():
    print(f"{ticker}: Train Sharpe = {res['Sharpe']:.2f}, Best mpfactor = {res['Best mpfactor']}")

print("\n=== Test Results ===")
for ticker, res in test_results.items():
    print(f"{ticker}: Test Sharpe = {res['Sharpe']:.2f}")

=== Train Results ===
BTCUSDT: Train Sharpe = 0.94, Best mpfactor = 1.5
ETHUSDT: Train Sharpe = 1.25, Best mpfactor = 1.9

=== Test Results ===
BTCUSDT: Test Sharpe = 0.44
ETHUSDT: Test Sharpe = 0.02
